In [ ]:
import cv2
import numpy as np

# ============================================================
# STEREO VISION DEPTH ESTIMATION USING OPENCV
# ============================================================
#
# Depth equation:
#
#                 f * B
#           Z = ---------
#                  d
#
# Z = depth/distance to object
# f = focal length in pixels
# B = baseline (distance between cameras)
# d = disparity in pixels
#
# ============================================================


# ------------------------------------------------------------
# 1. CAMERA PARAMETERS
# ------------------------------------------------------------

# Example focal length in pixels
# Replace this with the value obtained from camera calibration.
focal_length = 600.0

# Distance between the left and right cameras in metres
baseline = 0.07       # 10 cm


# ------------------------------------------------------------
# 2. LOAD LEFT AND RIGHT IMAGES
# ------------------------------------------------------------

left_img = cv2.imread("left.jpg")
right_img = cv2.imread("right.jpg")

if left_img is None or right_img is None:
    print("ERROR: Could not load left.jpg or right.jpg")
    print("Make sure both images are in the same folder as this program.")
    exit()

# Images must have the same dimensions
if left_img.shape != right_img.shape:
    print("ERROR: Left and right images must have the same dimensions.")
    exit()


# ------------------------------------------------------------
# 3. CONVERT IMAGES TO GRAYSCALE
# ------------------------------------------------------------

left_gray = cv2.cvtColor(left_img, cv2.COLOR_BGR2GRAY)
right_gray = cv2.cvtColor(right_img, cv2.COLOR_BGR2GRAY)


# ------------------------------------------------------------
# 4. CREATE STEREO MATCHER
# ------------------------------------------------------------

# StereoSGBM is generally more accurate than StereoBM.

min_disparity = 0
num_disparities = 16 * 8       # Must be divisible by 16
block_size = 5

stereo = cv2.StereoSGBM_create(
    minDisparity=min_disparity,
    numDisparities=num_disparities,
    blockSize=block_size,

    P1=8 * block_size ** 2,
    P2=32 * block_size ** 2,

    disp12MaxDiff=1,
    uniquenessRatio=10,
    speckleWindowSize=100,
    speckleRange=2
)


# ------------------------------------------------------------
# 5. CALCULATE DISPARITY
# ------------------------------------------------------------

disparity = stereo.compute(left_gray, right_gray)

# OpenCV stores SGBM disparity multiplied by 16.
# Divide by 16 to obtain disparity in pixels.
disparity = disparity.astype(np.float32) / 16.0


# ------------------------------------------------------------
# 6. CALCULATE DEPTH
# ------------------------------------------------------------

# Avoid division by zero and invalid disparities.
depth = np.zeros_like(disparity, dtype=np.float32)

valid = disparity > 0

depth[valid] = (
    focal_length * baseline
) / disparity[valid]


# ------------------------------------------------------------
# 7. CREATE DISPARITY IMAGE FOR DISPLAY
# ------------------------------------------------------------

valid_disparity = np.maximum(disparity, 0)

disp_display = cv2.normalize(
    valid_disparity,
    None,
    0,
    255,
    cv2.NORM_MINMAX
)

disp_display = np.uint8(disp_display)

# Apply colour map to make depth differences easier to see.
disp_colour = cv2.applyColorMap(
    disp_display,
    cv2.COLORMAP_JET
)


# ------------------------------------------------------------
# 8. MOUSE FUNCTION TO DISPLAY DEPTH
# ------------------------------------------------------------

def show_depth(event, x, y, flags, param):

    if event == cv2.EVENT_LBUTTONDOWN:

        d = disparity[y, x]

        if d > 0:

            Z = depth[y, x]

            print("\n--------------------------")
            print("Pixel:", x, y)
            print("Disparity: %.2f pixels" % d)
            print("Depth: %.2f metres" % Z)
            print("--------------------------")

        else:
            print("Invalid disparity at this point.")


# ------------------------------------------------------------
# 9. DISPLAY RESULTS
# ------------------------------------------------------------

cv2.namedWindow("Left Image")
cv2.namedWindow("Disparity / Depth")

cv2.setMouseCallback(
    "Left Image",
    show_depth
)

cv2.imshow("Left Image", left_img)
cv2.imshow("Right Image", right_img)
cv2.imshow("Disparity / Depth", disp_colour)

print("========================================")
print("STEREO VISION DEPTH DEMONSTRATION")
print("========================================")
print()
print("Focal length :", focal_length, "pixels")
print("Baseline     :", baseline, "metres")
print()
print("Click on an object in the LEFT IMAGE")
print("to estimate its distance.")
print()
print("Press Q to quit.")
print("========================================")


# ------------------------------------------------------------
# 10. WAIT UNTIL USER PRESSES Q
# ------------------------------------------------------------

while True:

    key = cv2.waitKey(1) & 0xFF

    if key == ord('q'):
        break


# ------------------------------------------------------------
# 11. CLOSE WINDOWS
# ------------------------------------------------------------

cv2.destroyAllWindows()

STEREO VISION DEPTH DEMONSTRATION

Focal length : 600.0 pixels
Baseline     : 0.07 metres

Click on an object in the LEFT IMAGE
to estimate its distance.

Press Q to quit.
